<a href="https://colab.research.google.com/github/zhiyuguo-innis/MSSP607/blob/main/zhiyuguo-innis/MSSP607/Modules/Week_5/PARTICIPATION_ACTIVITY_Week_5_Merge_two_Pandas_dataframes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PARTICIPATION ACTIVITY Week 5: Merge two Pandas dataframes.

Combine daily stock data for Facebook, Apple, Amazon, Netflix, and Google into one DataFrame, clean the types, and reshape it.

*Modified from Molin, S. (2019). Hands-On Data Analysis with Pandas, Ch. 3.*

## 1. Setup: clone the GitHub repository into Colab

Repository: `zhiyuguo-innis/MSSP607`. The CSV files (`aapl.csv`, `amzn.csv`, `fb.csv`, `goog.csv`, `nflx.csv`) are in `zhiyuguo-innis/MSSP607/Modules/Week_5/` inside the repo.

In [ ]:
from pathlib import Path
import os

GITHUB_USER = 'zhiyuguo-innis'
REPO_NAME = 'MSSP607'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_path = Path('/content') / REPO_NAME
    if not repo_path.exists():
        # For a private repo use: https://<TOKEN>@github.com/<user>/<repo>.git
        !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git /content/{REPO_NAME}
    else:
        !git -C /content/{REPO_NAME} pull
    os.chdir(repo_path)

# The repo has an extra zhiyuguo-innis/MSSP607/ folder level inside it,
# so the CSVs are at <repo root>/zhiyuguo-innis/MSSP607/Modules/Week_5
DATA_DIR = Path('zhiyuguo-innis/MSSP607/Modules/Week_5')
print('Working directory:', Path.cwd())
print('Data files:', sorted(p.name for p in DATA_DIR.glob('*.csv')))

Cloning into '/content/MSSP607'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 260 (delta 17), reused 2 (delta 2), pack-reused 188 (from 1)
Receiving objects: 100% (260/260), 3.98 MiB | 15.09 MiB/s, done.
Resolving deltas: 100% (63/63), done.
Working directory: /content/MSSP607
Data files: ['Student Performance Data.csv', 'aapl.csv', 'amzn.csv', 'fb.csv', 'goog.csv', 'nflx.csv']


In [ ]:
import pandas as pd
pd.options.display.max_rows = 20

## 2b–c. Read each file and add a `ticker` column

Each file name is the ticker symbol, so the ticker can be taken from the file name (e.g. `aapl.csv` → `AAPL`).

In [ ]:
tickers = ['fb', 'aapl', 'amzn', 'nflx', 'goog']

frames = {}
for t in tickers:
    df = pd.read_csv(DATA_DIR / f'{t}.csv')
    df['ticker'] = t.upper()
    frames[t] = df
    print(f'{t.upper():5s} rows={len(df):5d}  columns={list(df.columns)}')

frames['aapl'].head()

FB    rows=  251  columns=['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']
AAPL  rows=  251  columns=['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']
AMZN  rows=  251  columns=['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']
NFLX  rows=  251  columns=['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']
GOOG  rows=  251  columns=['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597,AAPL
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018,AAPL
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766,AAPL


## 2d. Append the files into a single DataFrame

These files share the same columns, so we are **stacking** rows, not matching on a key. That makes `pd.concat()` the right tool (not `merge()`).

In [ ]:
faang = pd.concat(frames.values(), ignore_index=True)

# Validation: row count should equal the sum of the parts
assert len(faang) == sum(len(df) for df in frames.values())
print('Combined shape:', faang.shape)
print(faang['ticker'].value_counts())
faang.head()

Combined shape: (1255, 7)
ticker
FB      251
AAPL    251
AMZN    251
NFLX    251
GOOG    251
Name: count, dtype: int64


,date,open,high,low,close,volume,ticker
0,2018-01-02,177.68,181.58,177.5500,181.42,18151903,FB
1,2018-01-03,181.88,184.78,181.3300,184.67,16886563,FB
2,2018-01-04,184.90,186.21,184.0996,184.33,13880896,FB
3,2018-01-05,185.59,186.90,184.9300,186.85,13574535,FB
4,2018-01-08,187.20,188.90,186.3300,188.28,17994726,FB


## 2e. Save to CSV

*Note: the assignment names the file `faanf.csv` in step e and `fanng.csv` in step f — these look like typos for the same file. The name is kept in one variable so both steps use the same file.*

In [ ]:
OUTPUT_FILE = DATA_DIR / 'faanf.csv'
faang.to_csv(OUTPUT_FILE, index=False)
print('Saved to', OUTPUT_FILE)

Saved to zhiyuguo-innis/MSSP607/Modules/Week_5/faanf.csv


## 2f. Type conversion and sorting

Read the saved file back, convert `date` to `datetime` and `volume` to integers, then sort by `date` and `ticker`.

In [ ]:
faang = pd.read_csv(OUTPUT_FILE)
print('Before:')
print(faang.dtypes, end='\n\n')

faang['date'] = pd.to_datetime(faang['date'])
faang['volume'] = faang['volume'].astype(int)
faang = faang.sort_values(['date', 'ticker']).reset_index(drop=True)

print('After:')
print(faang.dtypes)
faang.head(10)

Before:
date       object
open      float64
high      float64
low       float64
close     float64
volume      int64
ticker     object
dtype: object

After:
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker            object
dtype: object


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-02,1172.0000,1190.0000,1170.5100,1189.0100,2694494,AMZN
2,2018-01-02,177.6800,181.5800,177.5500,181.4200,18151903,FB
3,2018-01-02,1048.3400,1066.9400,1045.2300,1065.0000,1237564,GOOG
4,2018-01-02,196.1000,201.6500,195.4200,201.0700,10966889,NFLX
5,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
6,2018-01-03,1188.3000,1205.4900,1188.3000,1204.2000,3108793,AMZN
7,2018-01-03,181.8800,184.7800,181.3300,184.6700,16886563,FB
8,2018-01-03,1064.3100,1086.2900,1063.2100,1082.4800,1430170,GOOG
9,2018-01-03,202.0500,206.2100,201.5000,205.0500,8591369,NFLX


## 2g. Seven rows with the highest volume

In [ ]:
faang.nlargest(7, 'volume')

,date,open,high,low,close,volume,ticker
712,2018-07-26,174.8900,180.1300,173.7500,176.2600,169803668,FB
267,2018-03-20,167.4700,170.2000,161.9500,168.1500,129851768,FB
287,2018-03-26,160.8200,161.1000,149.0200,160.0600,126116634,FB
272,2018-03-21,164.8000,173.4000,163.3000,169.3900,106598834,FB
910,2018-09-21,219.0727,219.6482,215.6097,215.9768,96246748,AAPL
1225,2018-12-21,156.1901,157.4845,148.9909,150.0862,95744384,AAPL
1060,2018-11-02,207.9295,211.9978,203.8414,205.8755,91328654,AAPL


## 2h. Melt to long format

`date` and `ticker` together identify one observation, so they are the `id_vars`. Every other column (`open`, `high`, `low`, `close`, `volume`) is melted into a `variable` / `value` pair.

In [ ]:
melted = faang.melt(id_vars=['date', 'ticker'],
                    var_name='variable', value_name='value')

# Validation: each original row becomes one row per melted column
n_value_cols = faang.shape[1] - 2
assert len(melted) == len(faang) * n_value_cols
print('Original shape:', faang.shape, '-> melted shape:', melted.shape)
print('Melted variables:', melted['variable'].unique())
melted.head(10)

Original shape: (1255, 7) -> melted shape: (6275, 4)
Melted variables: ['open' 'high' 'low' 'close' 'volume']


,date,ticker,variable,value
0,2018-01-02,AAPL,open,166.9271
1,2018-01-02,AMZN,open,1172.0000
2,2018-01-02,FB,open,177.6800
3,2018-01-02,GOOG,open,1048.3400
4,2018-01-02,NFLX,open,196.1000
5,2018-01-03,AAPL,open,169.2521
6,2018-01-03,AMZN,open,1188.3000
7,2018-01-03,FB,open,181.8800
8,2018-01-03,GOOG,open,1064.3100
9,2018-01-03,NFLX,open,202.0500


## 4. Push the notebook and data back to GitHub (from Colab)

Optional: if you edited in Colab, save a copy to the repo and push. Pushing needs a personal access token. Alternatively, use **File → Save a copy in GitHub** in Colab.

In [ ]:
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git add data/faanf.csv
# !git commit -m "Week 05: combined FAANG data"
# !git push https://<TOKEN>@github.com/{GITHUB_USER}/{REPO_NAME}.git